# L5 — Chatbot UI

## What you'll learn

- How customers sign in to the chatbot with Amazon Cognito
- How the chatbot passes each customer's ID to the orchestrator, so memory is kept per customer
- How the answer streams into the chat as it is written
- How a decision model triages each message: scope, escalation, urgency and sentiment

## Overview

This notebook launches the AnyCompany support chatbot, which ties the whole system together. Customers sign in with Cognito, and the orchestrator answers with the help of the specialist agents, keeping memory separate for each customer. If you connected a decision provider, the chatbot first triages each message and shows sentiment, urgency and escalation badges. The sidebar shows the technology you chose at each layer.

## Architecture

![Application layer: the customer's message from the chatbot is triaged by a decision model (Strands Decider or Perplexity Decisions). Out-of-scope messages get a canned reply; in-scope messages go to the AgentCore Harness, which uses the Order, Refund and Research agents, and the reply streams back.](images/l5-application-architecture.png)

This notebook launches the chatbot UI at the left of the diagram. The decision model and the agent loop come from the earlier notebooks.

## Steps

0. Set up: install the packages and load the configuration
1. Launch the chatbot, in SageMaker Studio or on your own machine

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- [`3_orchestrator_agent.ipynb`](../l3-orchestration/3_orchestrator_agent.ipynb) is complete.
- Optional: [`1_decision_models.ipynb`](1_decision_models.ipynb) for triage, and [`1_sensitive_data_masking.ipynb`](../l4-security/1_sensitive_data_masking.ipynb) for PII masking.

## Step 0: Set up

### 0.1 Install the packages

Install the packages for the chatbot.

In [ ]:
%pip install --quiet --upgrade boto3 streamlit==1.54.0 "pyjwt[crypto]==2.13.0"

### 0.2 Load the workshop configuration

Read the orchestrator and sign-in settings saved by earlier notebooks.

In [ ]:
import os
import uuid
import boto3
import jwt

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")  # Change to your preferred region
SSM_PREFIX = "/anycompany/agentcore"
ssm = boto3.client("ssm", region_name=REGION)

def get_ssm(name):
    return ssm.get_parameter(Name=f"{SSM_PREFIX}/{name}", WithDecryption=True)["Parameter"]["Value"]

try:
    HARNESS_ARN = get_ssm("harness_arn")
except ssm.exceptions.ParameterNotFound:
    raise RuntimeError("harness_arn is not in SSM. Run l3-orchestration/3_orchestrator_agent.ipynb first.") from None
COGNITO_CLIENT_ID = get_ssm("cognito_client_id")

print(f"Region:         {REGION}")
print(f"Harness ARN:    {HARNESS_ARN}")
print(f"Cognito client: {COGNITO_CLIENT_ID}")

## Step 1: Launch the chatbot

Sign in as `gold_customer`, `silver_customer` or `bronze_customer`. The password is in SSM Parameter Store at `/anycompany/agentcore/user_password`.

### 1.1 Launch in SageMaker Studio

Run the cell and open the link. Stop the app with the kernel's interrupt (■) button.

In [ ]:
from IPython.display import display, HTML

display(HTML(
    '<h3>👉 <a href="/jupyterlab/default/proxy/8502/" target="_blank">Launch the Chatbot UI</a></h3>'
    '<p>If the page does not load, wait a few seconds for Streamlit to start, then refresh.</p>'
))
!streamlit run chatbot_app.py --server.headless true --server.port 8502

### 1.2 Launch on your own machine (not for the Workshop Studio environment)

Use this only outside SageMaker Studio. Run the cell and open `http://127.0.0.1:8502`.

In [ ]:
!streamlit run chatbot_app.py --server.headless true --server.port 8502

## Summary

You launched the AnyCompany support chatbot. Customers sign in, each message is triaged by your decision model, and the orchestrator and the specialist agents answer. The sidebar shows the technology you chose at each layer.

**Next:** [`1_cloudwatch_omni.ipynb`](../l6-observability/1_cloudwatch_omni.ipynb) or [`1_datadog_agent_observability.ipynb`](../l6-observability/1_datadog_agent_observability.ipynb), depending on the observability provider you chose. Follow a conversation through its traces.